# Instance Checkpoint Sweep

This notebook runs true instance segmentation testing across Toy Mask2Former (DINOv3 backbone) and Graha Mask R-CNN checkpoints. For each checkpoint, it saves every selected sample's model input tensor, instance target, instance prediction, boxes, scores, and metrics to disk.

The implementation lives in `scripts/python/instance_seg/instance_checkpoint_sweep.py` so this notebook and the script workflow use the same logic.

## Setup

In [ ]:
from argparse import Namespace
from pathlib import Path
import sys
from datetime import datetime

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "instance_checkpoint_sweep.ipynb").exists():
    NOTEBOOK_DIR = (Path.cwd() / "notebooks" / "full_model").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
SCRIPTS_PY_DIR = REPO_ROOT / "scripts" / "python"
SCRIPTS_TASK_DIR = SCRIPTS_PY_DIR / "instance_seg"
for path in [SCRIPTS_TASK_DIR, SCRIPTS_PY_DIR, REPO_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from lfm.full_model.all_tasks.utils.utils import ensure_data_symlink
from instance_checkpoint_sweep import build_config, discover_checkpoints, run_sweep

## Config

Set `TOY_CHECKPOINT_DIR` and `GRAHA_CHECKPOINT_DIR` to the checkpoint folders from the 100-epoch true-instance comparison run.

In [ ]:
# Data paths
INPUT_ROOT_DIR = None  # Optional source directory for ./data symlink
DATA_ROOT = Path("/panfs/ccds02/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_inst_seg_v2")
SIMLINK_DEST = INPUT_ROOT_DIR

# Checkpoint directories. Update these for the completed run.
RUN_OUTPUT_DIR = Path("/explore/nobackup/people/ajkerr1/Lunar_FM/iseg_outputs/date_YYYY_MM_DD-time_HH_MM_SS")
TOY_CHECKPOINT_DIR = RUN_OUTPUT_DIR / "checkpoints" / "toy_model"
GRAHA_CHECKPOINT_DIR = RUN_OUTPUT_DIR / "checkpoints" / "full_model"
MODELS = ["toy", "graha"]

# Output root for sweep artifacts.
date_str = datetime.now().strftime("date_%Y_%m_%d-time_%H_%M_%S")
OUTPUT_ROOT = NOTEBOOK_DIR / "outputs" / "instance_checkpoint_sweep" / date_str

# Sweep controls
PREDICTION_SPLIT = "test"
MAX_SAMPLES = None  # Use e.g. 5 for a quick notebook smoke test.
MAX_CHECKPOINTS = None  # Use e.g. 2 for a quick notebook smoke test.

# Model/data settings matching the comparison run.
TARGET_SIZE = 256
BAND_FILTER = [0, 1, 2, 3, 4, 5, 6]
TOY_BATCH_SIZE = 2
GRAHA_BATCH_SIZE = 2
TOY_NUM_WORKERS = 4
GRAHA_NUM_WORKERS = 4
GRAHA_WAC_MODE = "new-wac"  # Use "vis-uv" to reuse pretrained 5-band vis + 2-band uv modalities.
GRAHA_VIS_UV_MERGE_METHOD = "mean"
GRAHA_STATS_BATCH_SIZE = 16
TOY_NORMALIZE_INPUTS = True
PREDICTION_SCORE_THRESHOLD = 0.5
MASK_SHIFT = (0, 0)

## Build Config

In [ ]:
ensure_data_symlink(SIMLINK_DEST, NOTEBOOK_DIR / "data")

args = Namespace(
    simlink_dest=SIMLINK_DEST,
    data_root=str(DATA_ROOT) if DATA_ROOT is not None else None,
    output_root=str(OUTPUT_ROOT),
    toy_checkpoint_dir=str(TOY_CHECKPOINT_DIR) if TOY_CHECKPOINT_DIR is not None else None,
    graha_checkpoint_dir=str(GRAHA_CHECKPOINT_DIR) if GRAHA_CHECKPOINT_DIR is not None else None,
    models=MODELS,
    target_size=TARGET_SIZE,
    band_filter=BAND_FILTER,
    max_samples=MAX_SAMPLES,
    toy_batch_size=TOY_BATCH_SIZE,
    toy_num_workers=TOY_NUM_WORKERS,
    toy_normalize_inputs=TOY_NORMALIZE_INPUTS,
    dino_checkpoint=None,
    graha_pretrain_dir=None,
    graha_wac_mode=GRAHA_WAC_MODE,
    graha_vis_uv_merge_method=GRAHA_VIS_UV_MERGE_METHOD,
    graha_stats_batch_size=GRAHA_STATS_BATCH_SIZE,
    graha_batch_size=GRAHA_BATCH_SIZE,
    graha_num_workers=GRAHA_NUM_WORKERS,
    graha_backbone_lr=5.0e-5,
    graha_head_lr=2.0e-4,
    graha_layer_decay=0.75,
    graha_weight_decay=0.05,
    graha_warmup_steps=500,
    graha_anchor_sizes=[[8], [16], [32], [64]],
    graha_anchor_aspect_ratios=[0.5, 1.0, 2.0],
    graha_score_threshold=0.5,
    prediction_split=PREDICTION_SPLIT,
    prediction_score_threshold=PREDICTION_SCORE_THRESHOLD,
    mask_shift=MASK_SHIFT,
    max_checkpoints=MAX_CHECKPOINTS,
    seed=42,
    verbose=False,
)

config = build_config(args)
print("Output root:", config.output_root)
print("Data root:", config.data_root)

## Checkpoint Discovery

In [ ]:
if config.toy_checkpoint_dir is not None and "toy" in config.models:
    toy_checkpoints = discover_checkpoints(config.toy_checkpoint_dir, max_checkpoints=config.max_checkpoints)
    print(f"Toy checkpoints: {len(toy_checkpoints)}")
    print(toy_checkpoints[:3])

if config.graha_checkpoint_dir is not None and "graha" in config.models:
    graha_checkpoints = discover_checkpoints(config.graha_checkpoint_dir, max_checkpoints=config.max_checkpoints)
    print(f"Graha checkpoints: {len(graha_checkpoints)}")
    print(graha_checkpoints[:3])

## Run Sweep

In [ ]:
results = run_sweep(config)
results.keys()